# CVS Recommendation System — Campaign Analytics Dashboard

**Purpose:** Comprehensive analysis of the two-tower recommendation system's data assets to answer the key strategic questions for a coupon + personalization campaign rollout.

**Grounded in real CVS Health 10-K filings:**
- **FY2024** (10-K filed Feb 12, 2025) and **FY2025** (10-K filed Feb 10, 2026)
- All financial context, revenue distributions, and projections reference actual SEC filings

**Data sources:** DuckDB analytics database (`cvs_analytics.duckdb`) containing:
- **8,983** real CVS store locations (CVS operates 8,979 as of Dec 31, 2025)
- **12,000** product catalog (OTC + Rx)
- **10M** synthetic customer profiles
- **16.4M** digital coupon clip events
- **10B** purchase transactions (pre-aggregated into feature tables)

---

## Table of Contents
1. [Setup & Data Connection](#1)
2. [Data Overview & Health Check](#2)
3. [Top 100 Target Stores / Markets](#3)
4. [Top 100 Products by Revenue](#4)
5. [Profit Margin Analysis](#5)
6. [Coupon Campaign Analysis](#6)
7. [Product Tier Strategy](#7)
8. [Customer Segmentation & Tiering](#8)
9. [Inventory Optimization Strategy](#9)
10. [Deployment Strategy](#10)
11. [CVS Financial Reality — 10-K Grounding](#11)
12. [Revenue Impact: Conservative Projections](#12)
13. [Shrinkage & Loss Prevention Analysis](#13)
14. [Key Recommendations & Risk Factors](#14)

<a id='1'></a>
## 1. Setup & Data Connection

In [ ]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)

DB_PATH = str(Path("../data/db/cvs_analytics.duckdb").resolve())
con = duckdb.connect(DB_PATH, read_only=True)

# Tune for analytics
con.execute("SET memory_limit='16GB'")
con.execute("SET threads=4")

print(f"Connected to {DB_PATH}")
tables_q = "SELECT table_name FROM information_schema.tables WHERE table_type='BASE TABLE' ORDER BY table_name"
print(f"Tables: {[r[0] for r in con.execute(tables_q).fetchall()]}")

<a id='2'></a>
## 2. Data Overview & Health Check

Quick census of all data assets and key statistics.

In [ ]:
# Table census
tables = {
    "stores": "Real CVS locations",
    "products": "Product catalog (OTC + Rx)",
    "customers": "Synthetic customer profiles",
    "coupon_clips": "Digital coupon clip events",
    "product_features": "Per-product aggregated metrics (from 10B txns)",
    "customer_features": "Per-customer aggregated metrics (from 10B txns)",
    "product_tiers": "Products with coupon-based tier assignments",
}
census = []
for tbl, desc in tables.items():
    n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    census.append({"Table": tbl, "Rows": f"{n:,}", "Description": desc})

census_df = pd.DataFrame(census)
display(census_df.style.hide(axis='index').set_caption("Data Asset Census"))

In [ ]:
# Key aggregate stats (from pre-materialized feature tables — no transaction scan needed)
stats = con.execute("""
    SELECT
        (SELECT SUM(total_revenue) FROM product_features) AS total_revenue,
        (SELECT AVG(margin_pct) FROM product_features) AS avg_margin,
        (SELECT SUM(total_spend) FROM customer_features) AS total_customer_spend,
        (SELECT AVG(total_spend) FROM customer_features) AS avg_customer_spend,
        (SELECT AVG(total_transactions) FROM customer_features) AS avg_txns_per_customer,
        (SELECT COUNT(*) FROM coupon_clips) AS total_coupon_clips,
        (SELECT SUM(CASE WHEN redeemed THEN 1 ELSE 0 END) FROM coupon_clips) AS total_redeemed,
        (SELECT COUNT(DISTINCT loyalty_number) FROM coupon_clips) AS unique_clippers
""").fetchdf().T
stats.columns = ["Value"]
stats.index = [
    "Total Revenue (all products)",
    "Avg Product Margin %",
    "Total Customer Spend",
    "Avg Spend per Customer",
    "Avg Transactions per Customer",
    "Total Coupon Clips",
    "Total Coupons Redeemed",
    "Unique Coupon Clippers",
]
display(stats.style.format("{:,.2f}").set_caption("High-Level KPIs"))

<a id='3'></a>
## 3. Top 100 Target Stores / Markets

Transaction data uses real CVS store IDs that join directly to the 8,983-store location table. We combine per-store transaction performance with geographic data to identify the highest-value locations for campaign rollout.

In [ ]:
# State-level market view: store supply + customer demand
store_supply = con.execute("""
    SELECT state, COUNT(*) AS store_count
    FROM stores
    GROUP BY state
    ORDER BY store_count DESC
""").fetchdf()

customer_demand = con.execute("""
    SELECT
        state,
        COUNT(*) AS customers,
        SUM(total_spend) AS total_revenue,
        AVG(total_spend) AS avg_spend,
        AVG(total_transactions) AS avg_txns,
        SUM(coupon_clips_count) AS total_clips,
        AVG(coupon_engagement_score) AS avg_coupon_engagement
    FROM customer_features
    GROUP BY state
    ORDER BY total_revenue DESC
""").fetchdf()

# Merge supply + demand
market_df = customer_demand.merge(store_supply, on="state", how="left")
market_df["revenue_per_store"] = market_df["total_revenue"] / market_df["store_count"]
market_df["customers_per_store"] = market_df["customers"] / market_df["store_count"]

print("Top 25 markets by total revenue:")
display(market_df.head(25)[["state", "store_count", "customers", "total_revenue",
                             "avg_spend", "revenue_per_store", "customers_per_store",
                             "avg_coupon_engagement"]].style.format({
    "total_revenue": "${:,.0f}",
    "avg_spend": "${:,.2f}",
    "revenue_per_store": "${:,.0f}",
    "customers_per_store": "{:,.0f}",
    "avg_coupon_engagement": "{:.4f}",
}).set_caption("Top 25 Markets by Revenue"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top20 = market_df.head(20)

# Revenue by state
axes[0].barh(top20["state"][::-1], top20["total_revenue"][::-1] / 1e9, color=sns.color_palette("Blues_r", 20))
axes[0].set_xlabel("Total Revenue ($B)")
axes[0].set_title("Top 20 Markets by Total Revenue")
axes[0].xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.1fB'))

# Revenue per store (efficiency)
efficiency = market_df.nlargest(20, "revenue_per_store")
axes[1].barh(efficiency["state"][::-1], efficiency["revenue_per_store"][::-1] / 1e6,
             color=sns.color_palette("Greens_r", 20))
axes[1].set_xlabel("Revenue per Store ($M)")
axes[1].set_title("Top 20 Markets by Revenue per Store")
axes[1].xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.1fM'))

plt.tight_layout()
plt.show()

In [ ]:
# Top 100 individual CVS stores by customer revenue
# Uses customer_features joined to stores via state (fast, no txn scan)
# For direct per-store transaction totals, we'd scan the 10B txn table — 
# the query below uses market priority + store metadata for actionable targeting.
top100_stores = con.execute("""
    WITH market_rank AS (
        SELECT state, SUM(total_spend) AS state_revenue
        FROM customer_features
        GROUP BY state
    )
    SELECT
        s.store_id,
        s.name,
        s.city,
        s.state,
        s.zip_code,
        s.latitude,
        s.longitude,
        s.store_type,
        mr.state_revenue,
        ROW_NUMBER() OVER (PARTITION BY s.state ORDER BY s.store_id) AS rank_in_state
    FROM stores s
    JOIN market_rank mr ON s.state = mr.state
    ORDER BY mr.state_revenue DESC, s.store_id
    LIMIT 100
""").fetchdf()

print("Top 100 Target Store Locations (by market priority):")
display(top100_stores[["store_id", "name", "city", "state", "zip_code", "store_type",
                        "state_revenue"]].style.format({
    "state_revenue": "${:,.0f}"
}).set_caption("Top 100 CVS Store Locations for Campaign Rollout"))

<a id='4'></a>
## 4. Top 100 Products by Revenue

Ranked by total revenue from 10B transactions (pre-aggregated in `product_features`).

In [ ]:
top100_products = con.execute("""
    SELECT
        pf.product_id,
        p.name AS product_name,
        pf.brand,
        pf.category,
        pf.price,
        pf.unit_cost,
        pf.margin_pct,
        pf.total_units_sold,
        pf.total_revenue,
        pf.unique_buyers,
        pf.avg_discount_pct,
        pf.coupon_clips,
        pf.coupon_redemption_rate,
        pt.tier
    FROM product_features pf
    JOIN products p ON pf.product_id = p.product_id
    JOIN product_tiers pt ON pf.product_id = pt.product_id
    WHERE pf.is_rx = false
    ORDER BY pf.total_revenue DESC
    LIMIT 100
""").fetchdf()

print(f"Top 100 Products by Revenue (OTC only):")
display(top100_products[["product_id", "product_name", "brand", "category", "price",
                          "margin_pct", "total_revenue", "total_units_sold",
                          "unique_buyers", "tier"]].head(25).style.format({
    "price": "${:,.2f}",
    "margin_pct": "{:.1%}",
    "total_revenue": "${:,.0f}",
    "total_units_sold": "{:,.0f}",
    "unique_buyers": "{:,.0f}",
}).set_caption("Top 25 of 100 Products — Revenue Ranked"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Top 20 products by revenue
top20p = top100_products.head(20)
labels = [f"{row['brand'][:12]}\n({row['category'][:15]})" for _, row in top20p.iterrows()]
axes[0].barh(labels[::-1], top20p["total_revenue"].values[::-1] / 1e9,
             color=sns.color_palette("viridis", 20))
axes[0].set_xlabel("Total Revenue ($B)")
axes[0].set_title("Top 20 Products by Revenue")

# Revenue by category (all products)
cat_revenue = con.execute("""
    SELECT category, SUM(total_revenue) AS revenue, COUNT(*) AS products
    FROM product_features
    WHERE is_rx = false
    GROUP BY category
    ORDER BY revenue DESC
""").fetchdf()

axes[1].barh(cat_revenue["category"][::-1], cat_revenue["revenue"].values[::-1] / 1e9,
             color=sns.color_palette("rocket", len(cat_revenue)))
axes[1].set_xlabel("Total Revenue ($B)")
axes[1].set_title("Revenue by Product Category (OTC)")

plt.tight_layout()
plt.show()

<a id='5'></a>
## 5. Profit Margin Analysis

Margins calculated as `(price - unit_cost) / price`. This is the single most important metric for the coupon strategy — we can only afford to discount products with sufficient margin.

In [ ]:
# Margin by category
margin_by_cat = con.execute("""
    SELECT
        category,
        COUNT(*) AS product_count,
        AVG(margin_pct) AS avg_margin,
        MIN(margin_pct) AS min_margin,
        MAX(margin_pct) AS max_margin,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY margin_pct) AS p25,
        PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY margin_pct) AS median_margin,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY margin_pct) AS p75,
        SUM(total_revenue) AS total_revenue,
        SUM(total_revenue * margin_pct) AS total_gross_profit
    FROM product_features
    WHERE is_rx = false
    GROUP BY category
    ORDER BY avg_margin DESC
""").fetchdf()

display(margin_by_cat.style.format({
    "avg_margin": "{:.1%}", "min_margin": "{:.1%}", "max_margin": "{:.1%}",
    "p25": "{:.1%}", "median_margin": "{:.1%}", "p75": "{:.1%}",
    "total_revenue": "${:,.0f}", "total_gross_profit": "${:,.0f}",
}).set_caption("Profit Margins by Category (OTC Products)"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Margin distribution across all products
all_margins = con.execute("SELECT margin_pct FROM product_features WHERE is_rx = false").fetchdf()
axes[0].hist(all_margins["margin_pct"], bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(all_margins["margin_pct"].mean(), color="red", linestyle="--",
                label=f"Mean: {all_margins['margin_pct'].mean():.1%}")
axes[0].axvline(all_margins["margin_pct"].median(), color="orange", linestyle="--",
                label=f"Median: {all_margins['margin_pct'].median():.1%}")
axes[0].set_xlabel("Margin %")
axes[0].set_ylabel("Product Count")
axes[0].set_title("Distribution of Product Margins (OTC)")
axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.PercentFormatter(1.0))

# Avg margin by category (horizontal bar)
sorted_cats = margin_by_cat.sort_values("avg_margin")
colors = ["#e74c3c" if m < 0.40 else "#f39c12" if m < 0.50 else "#2ecc71"
          for m in sorted_cats["avg_margin"]]
axes[1].barh(sorted_cats["category"], sorted_cats["avg_margin"], color=colors)
axes[1].set_xlabel("Average Margin")
axes[1].set_title("Average Margin by Category")
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[1].axvline(0.45, color="gray", linestyle=":", alpha=0.5, label="45% threshold")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Per-product margin for top 100 products
fig, ax = plt.subplots(figsize=(16, 6))

margin_colors = ["#e74c3c" if m < 0.35 else "#f39c12" if m < 0.50 else "#2ecc71"
                 for m in top100_products["margin_pct"]]
ax.bar(range(100), top100_products["margin_pct"], color=margin_colors, alpha=0.8)
ax.set_xlabel("Product Rank (by revenue)")
ax.set_ylabel("Margin %")
ax.set_title("Profit Margin for Top 100 Revenue Products")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.axhline(0.45, color="gray", linestyle=":", alpha=0.7, label="45% discount safety line")
ax.legend()
plt.tight_layout()
plt.show()

low_margin = (top100_products["margin_pct"] < 0.35).sum()
print(f"\nOf the top 100 revenue products: {low_margin} have margin < 35% (risky to discount)")
print(f"Average margin of top 100: {top100_products['margin_pct'].mean():.1%}")

<a id='6'></a>
## 6. Coupon Campaign Analysis

Analysis of the existing digital coupon program: 16.4M clip events across ~1.2M active clippers.

In [ ]:
# Overall coupon stats
coupon_stats = con.execute("""
    SELECT
        COUNT(*) AS total_clips,
        SUM(CASE WHEN redeemed THEN 1 ELSE 0 END) AS total_redeemed,
        ROUND(SUM(CASE WHEN redeemed THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS redemption_rate_pct,
        COUNT(DISTINCT loyalty_number) AS unique_clippers,
        COUNT(DISTINCT product_id) AS unique_products_clipped,
        AVG(discount_value) AS avg_discount_value
    FROM coupon_clips
""").fetchdf()
display(coupon_stats.T.rename(columns={0: "Value"}).style.format("{:,.2f}").set_caption("Coupon Program Overview"))

In [ ]:
# By discount type
by_type = con.execute("""
    SELECT
        discount_type,
        COUNT(*) AS clips,
        SUM(CASE WHEN redeemed THEN 1 ELSE 0 END) AS redeemed,
        ROUND(SUM(CASE WHEN redeemed THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS redemption_rate_pct,
        AVG(discount_value) AS avg_discount_value,
        COUNT(DISTINCT loyalty_number) AS unique_clippers
    FROM coupon_clips
    GROUP BY discount_type
    ORDER BY clips DESC
""").fetchdf()

display(by_type.style.format({
    "clips": "{:,.0f}", "redeemed": "{:,.0f}",
    "redemption_rate_pct": "{:.1f}%", "avg_discount_value": "{:.2f}",
    "unique_clippers": "{:,.0f}",
}).set_caption("Coupon Performance by Discount Type"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Clip distribution by type
axes[0].pie(by_type["clips"], labels=by_type["discount_type"], autopct="%1.1f%%",
            colors=sns.color_palette("Set2", 3), startangle=90)
axes[0].set_title("Coupon Clips by Type")

# Redemption rate by type
colors = ["#2ecc71", "#3498db", "#e74c3c"]
axes[1].bar(by_type["discount_type"], by_type["redemption_rate_pct"], color=colors)
axes[1].set_ylabel("Redemption Rate (%)")
axes[1].set_title("Redemption Rate by Discount Type")
for i, v in enumerate(by_type["redemption_rate_pct"]):
    axes[1].text(i, v + 0.5, f"{v:.1f}%", ha="center", fontweight="bold")

# Monthly clip volume trend
monthly_clips = con.execute("""
    SELECT
        DATE_TRUNC('month', clip_date) AS month,
        COUNT(*) AS clips,
        SUM(CASE WHEN redeemed THEN 1 ELSE 0 END) AS redeemed
    FROM coupon_clips
    GROUP BY DATE_TRUNC('month', clip_date)
    ORDER BY month
""").fetchdf()
axes[2].plot(monthly_clips["month"], monthly_clips["clips"] / 1000, marker="o", label="Clips", markersize=3)
axes[2].plot(monthly_clips["month"], monthly_clips["redeemed"] / 1000, marker="s", label="Redeemed", markersize=3)
axes[2].set_ylabel("Count (thousands)")
axes[2].set_title("Monthly Coupon Activity")
axes[2].legend()
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Top 25 most-clipped products
top_clipped = con.execute("""
    SELECT
        cc.product_id,
        p.name AS product_name,
        p.brand,
        p.category,
        p.price,
        pf.margin_pct,
        COUNT(*) AS total_clips,
        SUM(CASE WHEN cc.redeemed THEN 1 ELSE 0 END) AS redeemed,
        ROUND(SUM(CASE WHEN cc.redeemed THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS redemption_rate,
        pt.tier
    FROM coupon_clips cc
    JOIN products p ON cc.product_id = p.product_id
    JOIN product_features pf ON cc.product_id = pf.product_id
    JOIN product_tiers pt ON cc.product_id = pt.product_id
    GROUP BY cc.product_id, p.name, p.brand, p.category, p.price, pf.margin_pct, pt.tier
    ORDER BY total_clips DESC
    LIMIT 25
""").fetchdf()

display(top_clipped.style.format({
    "price": "${:,.2f}", "margin_pct": "{:.1%}",
    "total_clips": "{:,.0f}", "redeemed": "{:,.0f}",
    "redemption_rate": "{:.1f}%",
}).set_caption("Top 25 Most-Clipped Products"))

### Coupon Campaign Recommendations

Based on the analysis above:

| Metric | Current | Opportunity |
|--------|---------|-------------|
| Active clippers | ~12% of customers | Target 20-25% with personalized push notifications |
| Dollar-off redemption | ~45% | Highest-performing type — expand for high-margin products |
| BOGO redemption | ~25% | Lowest — consider restricting to clearance/seasonal only |
| Percent-off volume | ~50% of all clips | Most common but mid-tier redemption — optimize thresholds |

**Key insight:** Dollar-off coupons redeem at 45% vs percent-off at 35%. For the recommendation system, we should default to **dollar-off** for targeted campaigns on high-margin products, and reserve percent-off for broad awareness campaigns.

<a id='7'></a>
## 7. Product Tier Strategy

Products are classified into 5 tiers based on coupon engagement and organic sales velocity. Each tier has a distinct recommended action:

| Tier | Definition | Action |
|------|------------|--------|
| **coupon_loyalist** | High clip count + high redemption (≥40%) | REDUCE discount 5-10%. They'll buy anyway. Protect margin. |
| **coupon_curious** | Moderate clips but low redemption (<40%) | OPTIMIZE discount type/value. Convert clips → purchases. |
| **organic_star** | High sales volume, low coupon engagement | DO NOT discount. Ensure inventory. Discounts destroy margin. |
| **hidden_gem** | High margin (≥45%) but low sales volume | LAUNCH targeted campaign. Start with 15-20% off to high-affinity customers. |
| **unclassified** | Doesn't fit above criteria | Analyze further. May need category-specific strategy. |

In [ ]:
# Tier distribution
tier_dist = con.execute("""
    SELECT
        tier,
        COUNT(*) AS product_count,
        AVG(margin_pct) AS avg_margin,
        AVG(total_revenue) AS avg_revenue,
        SUM(total_revenue) AS total_revenue,
        AVG(coupon_clips) AS avg_clips,
        AVG(coupon_redemption_rate) AS avg_redemption_rate,
        AVG(organic_purchase_ratio) AS avg_organic_ratio
    FROM product_tiers
    WHERE is_rx = false
    GROUP BY tier
    ORDER BY total_revenue DESC
""").fetchdf()

display(tier_dist.style.format({
    "product_count": "{:,.0f}",
    "avg_margin": "{:.1%}", "avg_revenue": "${:,.0f}",
    "total_revenue": "${:,.0f}", "avg_clips": "{:,.1f}",
    "avg_redemption_rate": "{:.1%}", "avg_organic_ratio": "{:.1%}",
}).set_caption("Product Tier Summary (OTC)"))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

tier_colors = {
    "coupon_loyalist": "#27ae60",
    "coupon_curious": "#f1c40f",
    "organic_star": "#3498db",
    "hidden_gem": "#9b59b6",
    "unclassified": "#95a5a6",
}
colors = [tier_colors.get(t, "gray") for t in tier_dist["tier"]]

# Product count by tier
axes[0].bar(tier_dist["tier"], tier_dist["product_count"], color=colors)
axes[0].set_ylabel("Product Count")
axes[0].set_title("Products per Tier")
axes[0].tick_params(axis='x', rotation=30)

# Revenue by tier
axes[1].bar(tier_dist["tier"], tier_dist["total_revenue"] / 1e9, color=colors)
axes[1].set_ylabel("Revenue ($B)")
axes[1].set_title("Total Revenue by Tier")
axes[1].tick_params(axis='x', rotation=30)

# Avg margin by tier
axes[2].bar(tier_dist["tier"], tier_dist["avg_margin"], color=colors)
axes[2].set_ylabel("Avg Margin")
axes[2].set_title("Average Margin by Tier")
axes[2].yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
axes[2].tick_params(axis='x', rotation=30)
axes[2].axhline(0.45, color="red", linestyle=":", alpha=0.5, label="45% line")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: coupon engagement vs organic purchase ratio, colored by tier
tier_scatter = con.execute("""
    SELECT coupon_clip_rate, organic_purchase_ratio, margin_pct, tier,
           total_revenue, total_units_sold
    FROM product_tiers
    WHERE is_rx = false AND total_units_sold > 0
""").fetchdf()

fig, ax = plt.subplots(figsize=(12, 7))
for tier, group in tier_scatter.groupby("tier"):
    ax.scatter(group["coupon_clip_rate"], group["organic_purchase_ratio"],
              s=group["margin_pct"] * 80, alpha=0.5,
              color=tier_colors.get(tier, "gray"), label=tier, edgecolors="white", linewidth=0.3)
ax.set_xlabel("Coupon Clip Rate (clips / unique buyers)")
ax.set_ylabel("Organic Purchase Ratio")
ax.set_title("Product Positioning: Coupon Engagement vs Organic Sales\n(bubble size = margin)")
ax.legend(title="Tier", loc="upper right")
plt.tight_layout()
plt.show()

<a id='8'></a>
## 8. Customer Segmentation & Tiering

We segment the 10M customer base along two dimensions:
1. **Spend tier** — based on total lifetime spend
2. **Coupon engagement tier** — based on coupon clip/redemption behavior

These combine into an actionable customer matrix.

In [ ]:
# Customer spend distribution
spend_stats = con.execute("""
    SELECT
        PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY total_spend) AS p10,
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY total_spend) AS p25,
        PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY total_spend) AS median,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY total_spend) AS p75,
        PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY total_spend) AS p90,
        PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY total_spend) AS p95,
        PERCENTILE_CONT(0.99) WITHIN GROUP (ORDER BY total_spend) AS p99,
        AVG(total_spend) AS mean,
        MAX(total_spend) AS max_spend
    FROM customer_features
""").fetchdf().T
spend_stats.columns = ["Value"]
display(spend_stats.style.format("${:,.2f}").set_caption("Customer Spend Distribution"))

In [ ]:
# Build customer segments using SQL (no need to pull 10M rows into Python)
customer_segments = con.execute("""
    WITH thresholds AS (
        SELECT
            PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY total_spend) AS spend_median,
            PERCENTILE_CONT(0.80) WITHIN GROUP (ORDER BY total_spend) AS spend_p80,
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY total_spend) AS spend_p95
        FROM customer_features
    ),
    classified AS (
        SELECT
            cf.*,
            CASE
                WHEN total_spend >= (SELECT spend_p95 FROM thresholds) THEN 'Platinum'
                WHEN total_spend >= (SELECT spend_p80 FROM thresholds) THEN 'Gold'
                WHEN total_spend >= (SELECT spend_median FROM thresholds) THEN 'Silver'
                ELSE 'Bronze'
            END AS spend_tier,
            CASE
                WHEN coupon_clips_count > 0 AND coupon_redemption_rate >= 0.40 THEN 'Active Redeemer'
                WHEN coupon_clips_count > 0 AND coupon_redemption_rate < 0.40 THEN 'Clipper (Low Convert)'
                ELSE 'Non-Clipper'
            END AS coupon_tier
        FROM customer_features cf
    )
    SELECT
        spend_tier,
        coupon_tier,
        COUNT(*) AS customers,
        AVG(total_spend) AS avg_spend,
        AVG(total_transactions) AS avg_txns,
        AVG(avg_basket_size) AS avg_basket,
        AVG(coupon_clips_count) AS avg_clips,
        AVG(coupon_redemption_rate) AS avg_redemption_rate
    FROM classified
    GROUP BY spend_tier, coupon_tier
    ORDER BY
        CASE spend_tier WHEN 'Platinum' THEN 1 WHEN 'Gold' THEN 2 WHEN 'Silver' THEN 3 ELSE 4 END,
        coupon_tier
""").fetchdf()

display(customer_segments.style.format({
    "customers": "{:,.0f}", "avg_spend": "${:,.2f}",
    "avg_txns": "{:,.1f}", "avg_basket": "${:,.2f}",
    "avg_clips": "{:,.1f}", "avg_redemption_rate": "{:.1%}",
}).set_caption("Customer Segment Matrix: Spend Tier x Coupon Engagement"))

In [ ]:
# Visualize the segment matrix as a heatmap
pivot_count = customer_segments.pivot_table(
    index="spend_tier", columns="coupon_tier", values="customers", fill_value=0
)
# Reorder
tier_order = ["Platinum", "Gold", "Silver", "Bronze"]
pivot_count = pivot_count.reindex(tier_order)

pivot_spend = customer_segments.pivot_table(
    index="spend_tier", columns="coupon_tier", values="avg_spend", fill_value=0
)
pivot_spend = pivot_spend.reindex(tier_order)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.heatmap(pivot_count / 1e6, annot=True, fmt=".2f", cmap="YlOrRd", ax=axes[0])
axes[0].set_title("Customer Count (millions)")
axes[0].set_ylabel("Spend Tier")

sns.heatmap(pivot_spend, annot=True, fmt=",.0f", cmap="YlGnBu", ax=axes[1])
axes[1].set_title("Avg Spend per Customer")
axes[1].set_ylabel("Spend Tier")

plt.tight_layout()
plt.show()

In [ ]:
# Top 100 customers by total spend
top100_customers = con.execute("""
    SELECT
        cf.customer_id,
        c.loyalty_number,
        cf.age,
        cf.gender,
        cf.state,
        cf.is_student,
        cf.total_spend,
        cf.total_transactions,
        cf.avg_basket_size,
        cf.coupon_clips_count,
        cf.coupon_redeemed_count,
        cf.coupon_redemption_rate,
        cf.coupon_engagement_score
    FROM customer_features cf
    JOIN customers c ON cf.customer_id = c.customer_id
    ORDER BY cf.total_spend DESC
    LIMIT 100
""").fetchdf()

print("Top 100 Customers by Lifetime Spend:")
display(top100_customers.head(25).style.format({
    "total_spend": "${:,.2f}",
    "avg_basket_size": "${:,.2f}",
    "coupon_redemption_rate": "{:.1%}",
    "coupon_engagement_score": "{:.4f}",
}).set_caption("Top 25 of 100 Highest-Value Customers"))

In [ ]:
# Demographic breakdown of top customers
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution of top 100 vs all
all_ages = con.execute("SELECT age FROM customer_features USING SAMPLE 100000").fetchdf()
axes[0].hist(all_ages["age"], bins=30, alpha=0.5, label="All customers (sample)", density=True, color="gray")
axes[0].hist(top100_customers["age"], bins=15, alpha=0.7, label="Top 100", density=True, color="#e74c3c")
axes[0].set_xlabel("Age")
axes[0].set_title("Age Distribution: Top 100 vs All")
axes[0].legend()

# State distribution of top 100
state_counts = top100_customers["state"].value_counts().head(15)
axes[1].barh(state_counts.index[::-1], state_counts.values[::-1], color="steelblue")
axes[1].set_xlabel("Count")
axes[1].set_title("Top 100 Customers by State")

# Gender distribution
gender_counts = top100_customers["gender"].value_counts()
axes[2].pie(gender_counts, labels=gender_counts.index, autopct="%1.1f%%",
            colors=sns.color_palette("Set2", len(gender_counts)))
axes[2].set_title("Top 100 Customers by Gender")

plt.tight_layout()
plt.show()

In [ ]:
# Spend tier summary
tier_summary = con.execute("""
    WITH thresholds AS (
        SELECT
            PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY total_spend) AS spend_median,
            PERCENTILE_CONT(0.80) WITHIN GROUP (ORDER BY total_spend) AS spend_p80,
            PERCENTILE_CONT(0.95) WITHIN GROUP (ORDER BY total_spend) AS spend_p95
        FROM customer_features
    ),
    classified AS (
        SELECT
            total_spend,
            total_transactions,
            coupon_clips_count,
            CASE
                WHEN total_spend >= (SELECT spend_p95 FROM thresholds) THEN 'Platinum'
                WHEN total_spend >= (SELECT spend_p80 FROM thresholds) THEN 'Gold'
                WHEN total_spend >= (SELECT spend_median FROM thresholds) THEN 'Silver'
                ELSE 'Bronze'
            END AS spend_tier
        FROM customer_features
    )
    SELECT
        spend_tier,
        COUNT(*) AS customers,
        SUM(total_spend) AS total_revenue,
        ROUND(SUM(total_spend) * 100.0 / (SELECT SUM(total_spend) FROM customer_features), 2) AS pct_of_revenue,
        AVG(total_spend) AS avg_spend,
        AVG(total_transactions) AS avg_txns,
        SUM(coupon_clips_count) AS total_clips
    FROM classified
    GROUP BY spend_tier
    ORDER BY CASE spend_tier WHEN 'Platinum' THEN 1 WHEN 'Gold' THEN 2 WHEN 'Silver' THEN 3 ELSE 4 END
""").fetchdf()

display(tier_summary.style.format({
    "customers": "{:,.0f}", "total_revenue": "${:,.0f}",
    "pct_of_revenue": "{:.1f}%", "avg_spend": "${:,.2f}",
    "avg_txns": "{:,.1f}", "total_clips": "{:,.0f}",
}).set_caption("Customer Spend Tier Summary"))

# Revenue concentration
print("\nRevenue Concentration:")
print(f"  Top 5% (Platinum): {tier_summary.iloc[0]['pct_of_revenue']:.1f}% of revenue")
print(f"  Top 20% (Platinum + Gold): {tier_summary.iloc[:2]['pct_of_revenue'].sum():.1f}% of revenue")

### Customer Segmentation Strategy

| Segment | Size | Strategy |
|---------|------|----------|
| **Platinum + Active Redeemer** | Smallest but highest value | VIP treatment. Personalized product recs. Minimize discounts — they already buy. Focus on cross-sell. |
| **Platinum + Non-Clipper** | High spenders who ignore coupons | Don't waste coupons. Use product recommendations only. These are organic loyalists. |
| **Gold + Clipper (Low Convert)** | Growing segment with potential | Optimize coupon type and timing. Switch to dollar-off. Add urgency (shorter expiration). |
| **Silver/Bronze + Active Redeemer** | Price-sensitive but engaged | Targeted coupons on high-margin products. Goal: increase basket size and frequency. |
| **Bronze + Non-Clipper** | Largest segment, lowest value | Broad awareness campaigns. Push app downloads. Goal: activate into coupon program. |

<a id='9'></a>
## 9. Inventory Optimization Strategy

The recommendation system creates predictable demand signals. Products recommended to high-affinity customers will see increased purchase rates — we need to ensure these items are stocked appropriately.

In [ ]:
# Products with high velocity + high margin = priority stock
inventory_priority = con.execute("""
    SELECT
        pt.product_id,
        p.name AS product_name,
        pt.brand,
        pt.category,
        pt.price,
        pt.margin_pct,
        pt.total_units_sold,
        pt.total_revenue,
        pt.unique_buyers,
        pt.coupon_clips AS coupon_clips,
        pt.coupon_redemption_rate,
        pt.tier,
        -- Priority score: high units * high margin * (1 + coupon boost)
        pt.total_units_sold * pt.margin_pct * (1 + pt.coupon_clip_rate * 0.5) AS inventory_priority_score
    FROM product_tiers pt
    JOIN products p ON pt.product_id = p.product_id
    WHERE pt.is_rx = false
    ORDER BY inventory_priority_score DESC
    LIMIT 50
""").fetchdf()

print("Top 50 Products for Inventory Priority (high velocity + high margin + coupon uplift potential):")
display(inventory_priority[["product_id", "product_name", "category", "price", "margin_pct",
                             "total_units_sold", "coupon_clips", "tier",
                             "inventory_priority_score"]].head(25).style.format({
    "price": "${:,.2f}", "margin_pct": "{:.1%}",
    "total_units_sold": "{:,.0f}", "coupon_clips": "{:,.0f}",
    "inventory_priority_score": "{:,.0f}",
}).set_caption("Inventory Priority: Top 25 of 50"))

In [ ]:
# Inventory strategy by tier
inv_by_tier = con.execute("""
    SELECT
        tier,
        COUNT(*) AS products,
        AVG(total_units_sold) AS avg_velocity,
        SUM(total_units_sold) AS total_units,
        AVG(margin_pct) AS avg_margin,
        AVG(coupon_clips) AS avg_coupon_clips,
        AVG(coupon_redemption_rate) AS avg_redemption
    FROM product_tiers
    WHERE is_rx = false
    GROUP BY tier
    ORDER BY total_units DESC
""").fetchdf()

display(inv_by_tier.style.format({
    "products": "{:,.0f}", "avg_velocity": "{:,.0f}",
    "total_units": "{:,.0f}", "avg_margin": "{:.1%}",
    "avg_coupon_clips": "{:,.1f}", "avg_redemption": "{:.1%}",
}).set_caption("Inventory Strategy by Product Tier"))

In [ ]:
# Hidden gems: high margin, low sales — biggest upside from recommendations
hidden_gems = con.execute("""
    SELECT
        pt.product_id,
        p.name AS product_name,
        pt.brand,
        pt.category,
        pt.price,
        pt.margin_pct,
        pt.total_units_sold,
        pt.total_revenue,
        pt.unique_buyers,
        pt.coupon_clips,
        -- Upside potential: margin room * price * gap to category average
        pt.margin_pct * pt.price AS dollar_margin
    FROM product_tiers pt
    JOIN products p ON pt.product_id = p.product_id
    WHERE pt.tier = 'hidden_gem'
    ORDER BY (pt.margin_pct * pt.price) DESC
    LIMIT 30
""").fetchdf()

print(f"Hidden Gems ({len(hidden_gems)} of {inv_by_tier[inv_by_tier['tier']=='hidden_gem']['products'].values[0]:,.0f} total):")
print("These products have high margins but low sales — prime candidates for targeted recommendation campaigns.\n")
display(hidden_gems.style.format({
    "price": "${:,.2f}", "margin_pct": "{:.1%}",
    "total_units_sold": "{:,.0f}", "total_revenue": "${:,.0f}",
    "unique_buyers": "{:,.0f}", "dollar_margin": "${:,.2f}",
}).set_caption("Top Hidden Gems by Dollar Margin"))

### Inventory Recommendations

| Tier | Inventory Action | Stock Level |
|------|-----------------|-------------|
| **organic_star** | **Never stockout.** These sell without discounts. Lost sale = pure lost margin. | +20% safety stock |
| **coupon_loyalist** | Stock for campaign spikes. Coupon drops drive predictable demand surges. | +15% during campaigns |
| **hidden_gem** | Start with modest stock increase. Ramp based on recommendation response. | +10% initially, monitor |
| **coupon_curious** | Standard stock. Conversion optimization may reduce volatility. | Maintain current |
| **unclassified** | Review for potential phase-out or repositioning. | Reduce by 5-10% |

<a id='10'></a>
## 10. Deployment Strategy

### Phased Rollout Plan

#### Phase 1: Model Training & Validation (Weeks 1-4)
- Train the two-tower model on 10B transaction history
- Generate customer embeddings (10M x 256-dim) and product embeddings (12K x 256-dim)
- Validate against holdout set — measure precision@K, NDCG, revenue-weighted metrics
- A/B test framework setup: 5% of customers in treatment, 5% in control

#### Phase 2: Pilot Markets (Weeks 5-8)
- Deploy to top 5 states by revenue (CA, TX, FL, NY, PA — see Section 3)
- Digital coupon push: personalized recommendations via CVS app
- Target: Platinum + Gold customers first (~25% of base, >50% of revenue)
- Focus on **hidden_gem** products (highest margin upside)

#### Phase 3: Scale & Optimize (Weeks 9-16)
- Expand to all 50 states
- Add Silver tier customers
- Introduce real-time recommendation updates (daily embedding refresh)
- Optimize coupon values based on tier strategy (Section 7)

#### Phase 4: Full Production (Week 17+)
- All customer segments active
- Automated inventory adjustment based on recommendation demand signals
- Continuous model retraining (weekly)
- Integration with in-store displays and email campaigns

### Infrastructure Requirements

| Component | Specification | Purpose |
|-----------|--------------|----------|
| **Embedding Store** | Custom VecDB (C + Apple Accelerate) or cloud equivalent | Top-K similarity queries in <10ms |
| **Feature Store** | DuckDB (2.3 GB) or cloud data warehouse | Real-time feature lookup |
| **Model Serving** | PyTorch model on GPU/MPS | Embedding generation for new customers |
| **Recommendation API** | REST endpoint, <100ms latency | Serve top-100 products per customer |
| **A/B Testing** | Feature flag system | Measure incremental revenue lift |

<a id='11'></a>
## 11. CVS Financial Reality — 10-K Grounding

All numbers below come directly from CVS Health Corporation's 10-K annual reports filed with the SEC. The FY2024 filing (period ending Dec 31, 2024) was filed Feb 12, 2025. The FY2025 filing (period ending Dec 31, 2025) was filed Feb 10, 2026.

**Why this matters:** Our recommendation system targets the Pharmacy & Consumer Wellness (P&CW) segment's **front store** business. Before projecting any revenue impact, we must understand the actual financial landscape we are operating within.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CVS HEALTH 10-K FINANCIAL DATA (DIRECTLY FROM SEC FILINGS)
# ═══════════════════════════════════════════════════════════════════
# Source: cvs-2025-excel.xls (FY2024, filed Feb 12, 2025)
#         cvs-2026-excel.xls (FY2025, filed Feb 10, 2026)

print("=" * 80)
print("CVS HEALTH CORPORATION -- 10-K FINANCIAL SUMMARY")
print("Source: SEC 10-K filings (FY2023, FY2024, FY2025)")
print("=" * 80)

header = f"{'':30s} {'FY2023':>12s} {'FY2024':>12s} {'FY2025':>12s}"

print("\n--- CONSOLIDATED REVENUE ($M) ---")
print(header)
print(f"{'Products':30s} {'$245,138':>12s} {'$231,521':>12s} {'$249,908':>12s}")
print(f"{'Premiums':30s} {'$99,192':>12s} {'$122,896':>12s} {'$134,751':>12s}")
print(f"{'Services':30s} {'$12,293':>12s} {'$16,239':>12s} {'$15,175':>12s}")
print(f"{'Net investment income':30s} {'$1,153':>12s} {'$2,153':>12s} {'$2,233':>12s}")
print(f"{'TOTAL REVENUES':30s} {'$357,776':>12s} {'$372,809':>12s} {'$402,067':>12s}")

print("\n--- SEGMENT REVENUE ($M) ---")
print(header)
print(f"{'Health Care Benefits':30s} {'$105,646':>12s} {'$130,665':>12s} {'$143,354':>12s}")
print(f"{'Health Services (PBM)':30s} {'$186,843':>12s} {'$173,605':>12s} {'$190,425':>12s}")
print(f"{'Pharmacy & Consumer Wellness':30s} {'$116,763':>12s} {'$124,500':>12s} {'$139,367':>12s}")
print(f"{'Intersegment Eliminations':30s} {'($51,927)':>12s} {'($56,412)':>12s} {'($71,563)':>12s}")

print("\n--- PHARMACY & CONSUMER WELLNESS DETAIL ($M) ---")
print("    (This is the retail segment our recommendation system targets)")
print(header)
print(f"{'Pharmacy revenue':30s} {'$92,111':>12s} {'$100,687':>12s} {'$115,510':>12s}")
print(f"{'Front Store revenue':30s} {'$22,458':>12s} {'$21,522':>12s} {'$21,459':>12s}")
print(f"{'Other':30s} {'$2,199':>12s} {'$2,291':>12s} {'$2,398':>12s}")
print(f"{'TOTAL P&CW':30s} {'$116,763':>12s} {'$124,500':>12s} {'$139,367':>12s}")
print(f"{'':30s}")
print(f"{'Pharmacy pct of segment':30s} {'78.9%':>12s} {'80.9%':>12s} {'82.9%':>12s}")
print(f"{'Front Store pct of segment':30s} {'21.1%':>12s} {'19.1%':>12s} {'17.1%':>12s}")

print("\n--- P&CW MARGINS & PROFITABILITY ---")
# Gross margin computed: Revenue - COGS / Revenue
# FY2023: ($116,763 - $91,447) / $116,763 = 21.7%
# FY2024: ($124,500 - $99,337) / $124,500 = 20.2%
# FY2025: ($139,367 - $113,583) / $139,367 = 18.5%
print(header)
print(f"{'Cost of products sold':30s} {'$91,447':>12s} {'$99,337':>12s} {'$113,583':>12s}")
print(f"{'Gross Profit':30s} {'$25,316':>12s} {'$25,163':>12s} {'$25,784':>12s}")
print(f"{'Gross Margin':30s} {'21.7%':>12s} {'20.2%':>12s} {'18.5%':>12s}")
print(f"{'Adjusted Operating Income':30s} {'$5,963':>12s} {'$5,774':>12s} {'$6,040':>12s}")
print(f"{'Adjusted OI Margin':30s} {'5.1%':>12s} {'4.6%':>12s} {'4.3%':>12s}")

print("\n--- SAME-STORE SALES ---")
print(header)
print(f"{'Pharmacy SSS':30s} {'+13.6%':>12s} {'+12.3%':>12s} {'+18.0%':>12s}")
print(f"{'Front Store SSS':30s} {'+0.3%':>12s} {'-2.1%':>12s} {'+1.2%':>12s}")
print(f"{'Rx volume growth':30s} {'+3.9%':>12s} {'+6.8%':>12s} {'+8.0%':>12s}")
print(f"{'Prescriptions filled (M)':30s} {'1,649.1':>12s} {'1,715.5':>12s} {'1,808.8':>12s}")

print("\n--- STORE COUNT ---")
print(header)
print(f"{'Beginning of year':30s} {'9,674':>12s} {'9,395':>12s} {'9,135':>12s}")
print(f"{'New/acquired':30s} {'39':>12s} {'39':>12s} {'87':>12s}")
print(f"{'Closed':30s} {'(318)':>12s} {'(299)':>12s} {'(243)':>12s}")
print(f"{'End of year':30s} {'9,395':>12s} {'9,135':>12s} {'8,979':>12s}")

print("\n--- EARNINGS ---")
print(header)
print(f"{'Net income ($M)':30s} {'$8,344':>12s} {'$4,614':>12s} {'$1,768':>12s}")
print(f"{'Diluted EPS':30s} {'$6.47':>12s} {'$3.66':>12s} {'$1.39':>12s}")

print("\n" + "=" * 80)
print("KEY TAKEAWAYS FOR OUR RECOMMENDATION SYSTEM:")
print("=" * 80)
takeaways = """
1. FRONT STORE IS SMALL AND SHRINKING
   - Front store: ~$21.5B of $124.5B P&CW revenue (17%)
   - Declined from $22.5B (FY2023) to $21.5B (FY2025)
   - Pharmacy grew from $92.1B to $115.5B over same period
   - Our recommendation system operates in the smaller, declining piece

2. GROSS MARGIN IS COMPRESSING: 21.7% -> 20.2% -> 18.5% over 3 years
   - Every dollar of margin we protect is increasingly valuable
   - Margin protection > incremental revenue in this environment

3. STORES ARE CLOSING: ~250-300 per year, from 9,674 to 8,979
   - Any per-store projection must account for a shrinking base

4. NET INCOME IS DECLINING: $8.3B -> $4.6B -> $1.8B
   - Aetna MBR went from 86.2% to 91.2%
   - $5.7B goodwill impairment in Health Services (FY2025)
   - Aggressive growth projections are NOT credible here

5. OUR SCOPE: OTC front store optimization only
   - We do NOT attempt to influence pharmacy revenue (~83% of segment)
   - We do NOT recommend increasing prescription volumes or prices
   - Our levers: coupon optimization, product placement, margin protection
"""
print(takeaways)

<a id='12'></a>
## 12. Revenue Impact: Conservative Projections

We deliberately use conservative projections. In the current economic climate with CVS facing margin compression, store closures, and declining front store traffic, **it is better to underestimate and over-deliver** than to set aggressive targets that erode credibility.

Our recommendation system targets the **front store** business only (~$21.5B). We do not attempt to influence pharmacy revenue. The key levers are:

1. **Coupon optimization** — better targeting, less waste on organic stars
2. **Hidden gem conversion** — driving sales to high-margin underperforming products
3. **Margin protection** — stop discounting products that sell without coupons
4. **Basket size** — cross-sell recommendations

We explicitly exclude pharmacy prescription revenue from all projections.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CONSERVATIVE REVENUE IMPACT MODEL
# Anchored to real CVS 10-K numbers
# ═══════════════════════════════════════════════════════════════════

# Real 10-K numbers (FY2024 P&CW segment, in millions)
REAL_FRONT_STORE_REVENUE = 21_522   # $21.522B front store (10-K TABLE11)
REAL_PHARMACY_REVENUE = 100_687     # $100.687B pharmacy (10-K TABLE11)
REAL_PCW_GROSS_MARGIN = 0.202       # 20.2% (10-K: ($124,500 - $99,337) / $124,500)
REAL_PCW_ADJ_OI_MARGIN = 0.046     # 4.6% adjusted operating margin
REAL_FRONT_STORE_SSS = -0.021      # -2.1% same-store sales FY2024
REAL_STORE_COUNT = 9_135            # End of FY2024 (10-K TABLE14)
REAL_STORES_CLOSING_PER_YEAR = 260  # Net closures FY2024

FRONT_STORE_GROSS_MARGIN_EST = REAL_PCW_GROSS_MARGIN

# Pull model baseline from synthetic data
baseline = con.execute("""
    SELECT SUM(total_revenue), SUM(total_revenue * margin_pct), AVG(margin_pct)
    FROM product_features WHERE is_rx = false
""").fetchone()
model_otc_revenue = baseline[0]
model_gross_profit = baseline[1]
model_avg_margin = baseline[2]

customer_base = con.execute("""
    SELECT COUNT(*), AVG(total_spend), AVG(total_transactions),
           AVG(avg_basket_size),
           SUM(CASE WHEN coupon_clips_count > 0 THEN 1 ELSE 0 END),
           AVG(CASE WHEN coupon_clips_count > 0 THEN coupon_redemption_rate ELSE NULL END)
    FROM customer_features
""").fetchone()
total_customers, avg_spend, avg_txns = customer_base[0], customer_base[1], customer_base[2]
avg_basket, coupon_active, avg_redemption = customer_base[3], customer_base[4], customer_base[5]

hidden_gem_potential = con.execute("""
    SELECT COUNT(*), SUM(total_revenue), AVG(margin_pct), AVG(total_units_sold)
    FROM product_tiers WHERE tier = 'hidden_gem' AND is_rx = false
""").fetchone()

organic_star_stats = con.execute("""
    SELECT COUNT(*), SUM(total_revenue), AVG(margin_pct), AVG(avg_discount_pct)
    FROM product_tiers WHERE tier = 'organic_star' AND is_rx = false
""").fetchone()

# Scale factor: real front store / model OTC revenue
scale_factor = (REAL_FRONT_STORE_REVENUE * 1e6) / model_otc_revenue

print("=== MODEL BASELINE (from synthetic data) ===")
print(f"Model OTC Revenue:          ${model_otc_revenue:,.0f}")
print(f"Model Gross Profit:         ${model_gross_profit:,.0f}")
print(f"Model Avg Margin:           {model_avg_margin:.1%}")
print(f"Total Customers:            {total_customers:,}")
print(f"Coupon-Active Customers:    {coupon_active:,} ({coupon_active/total_customers:.1%})")
print(f"Hidden Gems:                {hidden_gem_potential[0]:,} products, ${hidden_gem_potential[1]:,.0f} revenue")
print(f"Organic Stars:              {organic_star_stats[0]:,} products, ${organic_star_stats[1]:,.0f} revenue")
print(f"\nScale factor (real/model):   {scale_factor:.6f}")
print(f"Real front store revenue:    ${REAL_FRONT_STORE_REVENUE:,}M")

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# CONSERVATIVE SCENARIO MODEL
# No "optimistic" scenario -- not credible in this environment.
# ═══════════════════════════════════════════════════════════════════

scenarios = {}

for scenario, params in [
    ("Conservative", {
        "coupon_activation_lift": 0.02,
        "redemption_improvement": 0.03,
        "hidden_gem_velocity_lift": 0.10,
        "organic_star_discount_savings": 0.015,
        "basket_size_lift": 0.01,
        "frequency_lift": 0.005,
    }),
    ("Moderate", {
        "coupon_activation_lift": 0.05,
        "redemption_improvement": 0.06,
        "hidden_gem_velocity_lift": 0.20,
        "organic_star_discount_savings": 0.03,
        "basket_size_lift": 0.025,
        "frequency_lift": 0.015,
    }),
]:
    p = params

    new_clippers = total_customers * p["coupon_activation_lift"]
    coupon_activation_revenue = new_clippers * avg_spend * 0.03

    existing_clip_revenue = coupon_active * avg_spend * 0.02
    redemption_lift_revenue = existing_clip_revenue * p["redemption_improvement"]

    hidden_gem_revenue_lift = hidden_gem_potential[1] * p["hidden_gem_velocity_lift"]
    organic_margin_savings = organic_star_stats[1] * p["organic_star_discount_savings"]
    basket_lift_revenue = model_otc_revenue * p["basket_size_lift"]
    frequency_lift_revenue = model_otc_revenue * p["frequency_lift"]

    total_incremental = (
        coupon_activation_revenue + redemption_lift_revenue
        + hidden_gem_revenue_lift + organic_margin_savings
        + basket_lift_revenue + frequency_lift_revenue
    )
    total_real = total_incremental * scale_factor

    scenarios[scenario] = {
        "Coupon Activation": coupon_activation_revenue * scale_factor,
        "Redemption Improvement": redemption_lift_revenue * scale_factor,
        "Hidden Gem Uplift": hidden_gem_revenue_lift * scale_factor,
        "Margin Protection": organic_margin_savings * scale_factor,
        "Basket Size Lift": basket_lift_revenue * scale_factor,
        "Frequency Lift": frequency_lift_revenue * scale_factor,
        "Total Incremental Revenue": total_real,
        "% Lift over Front Store": total_real / (REAL_FRONT_STORE_REVENUE * 1e6) * 100,
        "Incremental Gross Profit (est)": total_real * FRONT_STORE_GROSS_MARGIN_EST,
    }

impact_df = pd.DataFrame(scenarios)
display(impact_df.style.format(
    "${:,.0f}", subset=pd.IndexSlice[[c for c in impact_df.index if c != "% Lift over Front Store"], :]
).format(
    "{:.2f}%", subset=pd.IndexSlice["% Lift over Front Store", :]
).set_caption("Revenue Impact: Conservative Projections (Real CVS Front Store Scale)"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

components = [k for k in list(scenarios.values())[0].keys()
              if k not in ("Total Incremental Revenue", "% Lift over Front Store",
                           "Incremental Gross Profit (est)")]
x = np.arange(len(scenarios))
bottom = np.zeros(len(scenarios))
colors_stack = sns.color_palette("Set2", len(components))

for i, comp in enumerate(components):
    values = [scenarios[s][comp] / 1e6 for s in scenarios]
    axes[0].bar(x, values, bottom=bottom, label=comp, color=colors_stack[i])
    bottom += values

axes[0].set_xticks(x)
axes[0].set_xticklabels(scenarios.keys())
axes[0].set_ylabel("Incremental Revenue ($M)")
axes[0].set_title("Revenue Lift Components by Scenario (Real $ Scale)")
axes[0].legend(loc="upper left", fontsize=8)

scenario_names = list(scenarios.keys())
real_baseline = REAL_FRONT_STORE_REVENUE
baseline_vals = [real_baseline] * len(scenarios)
incremental_vals = [scenarios[s]["Total Incremental Revenue"] / 1e6 for s in scenario_names]

axes[1].bar(x - 0.2, baseline_vals, 0.35, label="Baseline Front Store (No System)", color="#95a5a6")
axes[1].bar(x + 0.2, [b + i for b, i in zip(baseline_vals, incremental_vals)], 0.35,
            label="With Recommendation System", color="#27ae60")
axes[1].set_xticks(x)
axes[1].set_xticklabels(scenario_names)
axes[1].set_ylabel("Front Store Revenue ($M)")
axes[1].set_title(f"CVS Front Store: ${REAL_FRONT_STORE_REVENUE:,}M Baseline vs. With System")
axes[1].legend()

for i, s in enumerate(scenario_names):
    pct = scenarios[s]["% Lift over Front Store"]
    total = baseline_vals[i] + incremental_vals[i]
    axes[1].text(i + 0.2, total + 50, f"+{pct:.2f}%", ha="center", fontweight="bold", color="#27ae60")

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 80)
print("REVENUE IMPACT SUMMARY -- CONSERVATIVE PROJECTIONS")
print("Anchored to CVS Health 10-K (FY2024)")
print("=" * 80)
print(f"\nReal CVS Front Store Revenue (FY2024): ${REAL_FRONT_STORE_REVENUE:,}M")
print(f"Real P&CW Gross Margin:                 {REAL_PCW_GROSS_MARGIN:.1%}")
print(f"Front Store SSS Trend (FY2024):          {REAL_FRONT_STORE_SSS:+.1%}")
print(f"Store Count (End FY2024):                {REAL_STORE_COUNT:,}")
print(f"Net Closures/Year:                       ~{REAL_STORES_CLOSING_PER_YEAR}")
print()

for name, s in scenarios.items():
    print(f"--- {name} Scenario ---")
    incr = s["Total Incremental Revenue"]
    pct = s["% Lift over Front Store"]
    gp = s["Incremental Gross Profit (est)"]
    print(f"  Incremental Revenue:      ${incr:,.0f}")
    print(f"  Revenue Lift:             +{pct:.2f}% of front store")
    print(f"  Est. Incremental Profit:  ${gp:,.0f}")
    print()

print("IMPORTANT CONTEXT:")
print("  - These are CONSERVATIVE estimates. We intentionally underestimate.")
print("  - Front store SSS was -2.1% in FY2024 (recovered to +1.2% in FY2025)")
print("  - P&CW gross margin compressed from 21.7% to 18.5% over 3 years")
print("  - We target margin PROTECTION as much as revenue GROWTH")
print("  - Pharmacy revenue ($100.7B) is EXCLUDED from all projections")
print("  - Even 1% lift on $21.5B = $215M in incremental revenue")
print("  - Better to underestimate and over-deliver than vice versa")
print("  - In a declining-EPS environment ($6.47 -> $3.66 -> $1.39),")
print("    credible, conservative projections build investor trust")

<a id='13'></a>
## 13. Shrinkage & Loss Prevention Analysis

**Important caveat:** CVS Health does **not** disclose specific shrinkage or inventory loss figures in their 10-K filings. The numbers below are sourced from:
- Congressional testimony by CVS executives (~$200M/year in organized retail crime losses)
- Industry estimates (NRF reports retail shrinkage averaging 1.4-1.6% of sales)
- Analyst extrapolation from the deferred tax asset for inventory ($68M in FY2024, implying ~$262M in book/tax inventory adjustments at ~26% effective rate)

**We are transparent that these are estimates, not audited figures.** We include this analysis because loss prevention represents a defensive revenue play — reducing losses rather than generating new sales. This is a lower-risk, more predictable return than demand-side interventions.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# SHRINKAGE & LOSS PREVENTION ROI ANALYSIS
# ═══════════════════════════════════════════════════════════════════

# Estimated shrinkage (NOT from 10-K -- from public testimony & industry data)
SHRINK_RATE_LOW = 0.014   # 1.4% -- NRF low end
SHRINK_RATE_MID = 0.016   # 1.6% -- NRF average
ORC_DISCLOSED = 200       # $200M -- CVS executive congressional testimony

front_store_rev_M = REAL_FRONT_STORE_REVENUE  # $21,522M

estimated_shrinkage_low = front_store_rev_M * SHRINK_RATE_LOW
estimated_shrinkage_mid = front_store_rev_M * SHRINK_RATE_MID

print("=" * 80)
print("SHRINKAGE / INVENTORY LOSS ESTIMATES")
print("(NOT from 10-K -- sourced from public testimony & industry averages)")
print("=" * 80)
print(f"\nFront Store Revenue (FY2024):       ${front_store_rev_M:,.0f}M")
print(f"\nEstimated Total Shrinkage:")
print(f"  At 1.4% (NRF low):                ${estimated_shrinkage_low:,.0f}M")
print(f"  At 1.6% (NRF average):             ${estimated_shrinkage_mid:,.0f}M")
print(f"  Organized Retail Crime (testimony): ${ORC_DISCLOSED:,}M")
print(f"\nNote: Total shrinkage includes employee theft, admin errors,")
print(f"      vendor fraud, and shoplifting. ORC is a subset.")

# Loss Prevention Device ROI
print("\n" + "=" * 80)
print("LOSS PREVENTION DEVICE ROI MODEL")
print("=" * 80)

devices = [
    {"name": "EAS tags (disposable, high-theft items)",
     "cost_per_unit": 0.08, "units_needed": 5_000_000,
     "deterrence_rate": 0.60, "applicable_shrink_pct": 0.30},
    {"name": "Locked display cases (high-value)",
     "cost_per_unit": 350, "units_needed": 45_000,
     "deterrence_rate": 0.85, "applicable_shrink_pct": 0.15},
    {"name": "Self-checkout AI monitoring",
     "cost_per_unit": 15_000, "units_needed": 9_135,
     "deterrence_rate": 0.40, "applicable_shrink_pct": 0.10},
    {"name": "RFID inventory tracking",
     "cost_per_unit": 0.05, "units_needed": 20_000_000,
     "deterrence_rate": 0.25, "applicable_shrink_pct": 0.20},
]

est_shrinkage_M = estimated_shrinkage_mid
print(f"\nUsing midpoint shrinkage estimate: ${est_shrinkage_M:,.0f}M")
print(f"{'':45s} {'Cost':>10s} {'Target':>10s} {'Savings':>10s} {'Net':>10s} {'ROI':>8s}")
print(f"{'Device':45s} {'($M/yr)':>10s} {'Shrink':>10s} {'($M/yr)':>10s} {'($M/yr)':>10s} {'':>8s}")
print("-" * 95)

total_cost = 0
total_savings = 0

for d in devices:
    annual_cost = d["cost_per_unit"] * d["units_needed"] / 1e6
    applicable = est_shrinkage_M * d["applicable_shrink_pct"]
    savings = applicable * d["deterrence_rate"]
    net = savings - annual_cost
    roi_pct = (savings / annual_cost - 1) * 100 if annual_cost > 0 else 0
    total_cost += annual_cost
    total_savings += savings
    print(f"{d['name']:45s} ${annual_cost:>8.1f} {d['applicable_shrink_pct']:>9.0%}"
          f" ${savings:>8.1f} ${net:>8.1f} {roi_pct:>+7.0f}%")

print("-" * 95)
net_total = total_savings - total_cost
roi_total = (total_savings / total_cost - 1) * 100
print(f"{'TOTAL':45s} ${total_cost:>8.1f} {'':>10s}"
      f" ${total_savings:>8.1f} ${net_total:>8.1f} {roi_total:>+7.0f}%")

print(f"\nKey insight: Loss prevention is a DEFENSIVE play.")
print(f"  Total annual investment:  ${total_cost:,.1f}M")
print(f"  Expected loss mitigation: ${total_savings:,.1f}M")
print(f"  Net savings:              ${net_total:,.1f}M")
print(f"  This is margin protection, not revenue growth.")
print(f"  Think of it as a short position hedging retail losses.")
print(f"\nCAVEATS:")
print(f"  - Shrinkage estimates are NOT from the 10-K")
print(f"  - Device costs are industry averages, not CVS-specific")
print(f"  - Deterrence rates vary by location and implementation")
print(f"  - Locked cases can REDUCE sales 15-25% due to friction")
print(f"  - RFID savings mostly from admin error reduction, not theft")

<a id='14'></a>
## 14. Key Recommendations & Risk Factors

### Critical Path Items

1. **Train the two-tower model** — All data and features are ready. Training on M4 Max with MPS acceleration should complete in hours.

2. **Run inference** — Score all 10M customers x 12K products. Output: top-100 products per customer with tier-based actions.

3. **Focus on margin protection first** — In a 21.7% -> 18.5% gross margin compression environment, protecting existing margin is more valuable than incremental revenue. Stop discounting organic stars.

4. **Hidden gems second** — High-margin, low-velocity products are the cleanest incremental opportunity. No cannibalization risk.

5. **Conservative deployment** — Start with 500 stores, measure for 90 days, then expand. Do not project system-wide results from a pilot.

### Revenue Expectations (Grounded in 10-K Reality)

| Scenario | Front Store Lift | Incremental Revenue | Context |
|----------|-----------------|-------------------|---------|
| **Conservative** | ~1-2% | ~$200-400M | This is our target. Credible, achievable. |
| **Moderate** | ~3-5% | ~$600-1,000M | Requires strong execution across all levers. |

**We do not publish an "optimistic" scenario.** Given CVS's declining EPS ($6.47 -> $3.66 -> $1.39), margin compression, and store closures, any projection above moderate would not be credible to investors or the board.

### Risk Factors from the 10-K

The following risks are directly cited in CVS Health's 10-K filings:

| Risk | 10-K Reference | Impact on Our System |
|------|---------------|---------------------|
| **Pharmacy reimbursement pressure** | "Continued pharmacy reimbursement pressure" | Not our problem (front store only), but overall margin compression affects the business |
| **Store closures** | 695 net closures FY2023-FY2025 (9,674 -> 8,979) | Shrinking deployment base. Must account for ~250 fewer stores/year |
| **Front store traffic decline** | Front store SSS: +0.3% -> -2.1% -> +1.2% | Volatile. System may stabilize traffic but should not assume return to growth |
| **Competitive pressure** | "Highly competitive and evolving environment" | Amazon, Walmart, dollar stores compete for front store wallet share |
| **PBM regulatory risk** | FTC lawsuit, PBM transparency legislation | Does not directly affect front store recommendations |
| **Health Care Benefits losses** | MBR: 86.2% -> 92.5% -> 91.2%; HCB adj OI: $5.6B -> $307M -> $2.9B | Creates pressure that may limit investment in front store initiatives |
| **Goodwill impairment** | $5.725B impairment in Health Services (FY2025) | Signals acquisition underperformance. Capital allocation scrutiny will be high |
| **Net income decline** | $8.3B -> $4.6B -> $1.8B over 3 years | Any new initiative must show clear, conservative ROI |

### Strategy: Underestimate, Then Over-Deliver

The stock market has already priced in the decline ($6.47 -> $1.39 EPS). If we project conservative numbers and beat them:
1. The stock re-rates upward on earnings surprise
2. Management credibility increases with investors
3. The recommendation system gets more investment for expansion

If we project aggressive numbers and miss:
1. Further credibility erosion
2. The program gets cut in the next restructuring
3. We lose the organizational mandate

**The smart play is the conservative play.** A 1-2% lift on $21.5B in front store revenue = $215-430M. That is real money. It does not need to be $2B to matter.

### What Would Happen If We Did Nothing?

Without a recommendation system:
- **Coupon waste continues** — 65% of clips go unredeemed
- **Hidden gems stay hidden** — high-margin products remain undiscovered
- **Organic stars get discounted** — uniform promotions destroy margin on products that don't need help
- **No personalization** — every customer sees the same offers
- **Competitive gap widens** — retailers with recommendation systems continue pulling ahead

The recommendation system isn't just a revenue play — it's a **margin defense** play. In a compressing-margin environment, the biggest win comes not from selling more, but from selling smarter.

In [ ]:
# Close connection
con.close()
print("Database connection closed.")